In [17]:
import pandas as pd
import os
import numpy as np

In [18]:
def to_csv(df, path):
    # Prepend dtypes to the top of df (from https://stackoverflow.com/a/43408736/7607701)
    df.loc[-1] = df.dtypes
    df.index = df.index + 1
    df.sort_index(inplace=True)
    # Then save it to a csv
    df.to_csv(path, index=False)

def read_csv(path):
    # Read types first line of csv
    dtypes = pd.read_csv(path, nrows=1).iloc[0].to_dict()
    # Read the rest of the lines with the types from above
    return pd.read_csv(path, dtype=dtypes, skiprows=[1])

In [19]:
schedule_dtypes = {
    "No. of Veh": "int64",
    "Service interval": "string",   
    "Run time (min)": "int64",
    "Term time (min)": "int64",
    "Avg. spd (km/h)": "float64",
    "Time of Day (morning: 0-late evening: 4)*": "int64",
    "Weekday=0/Sat=1/Sun=2": "int64",
    "RT dist (km)": "float64",
    "Interruption": "int64",
    "EB/WB": "string",
    "bunch": "int64",
    "total delay": "float64",
    "gap": "int64"
}

schedule_cols = ["date", "time period start","time period end"] + list(schedule_dtypes.keys())

# read each csv with the correct dtypes
loc = '../data/schedule_data/processed_data/'
csv_file_names = [loc+file for file in os.listdir(loc) if file.startswith('schedule_data_sunday')]
list_of_dataframes = [pd.read_csv(file,usecols=schedule_cols,dtype=schedule_dtypes ) 
    for file in csv_file_names]
sun_df = pd.concat(list_of_dataframes, ignore_index=True)

# For datetime dtypes, these need to be parsed separately
sun_df["date"] = pd.to_datetime(sun_df["date"]).dt.floor("D")
sun_df["time period start"] = pd.to_datetime(sun_df["time period start"])
sun_df["time period end"] = pd.to_datetime(sun_df["time period end"])

In [20]:
weather_dtypes = {
    #"unixtime": "int64",
    #"period_index": "int64",
    "conditions": "string",
    "pop": "Int64",                
    "pop_category": "string",   
    "temperature": "float64",
    #"windchill": "float64",
    #"humidex": "float64",         
    "wind_direction": "string",  
    #"wind_gust": "float64",        
    "wind_speed": "float64",
}

weather_cols = ["date_time_local", "period_string"] + list(weather_dtypes.keys())

loc = '../data/raw_data/Weatherstats_data/'
weather_df = pd.read_csv(
    loc + "weatherstats_toronto_forecast_hourly.csv",
    usecols=weather_cols,
    dtype=weather_dtypes,
    parse_dates=["date_time_local", "period_string"]
)


/var/folders/57/j7359fzj3n7g0sxmfg939lb40000gn/T/ipykernel_70067/4273709804.py:18: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  weather_df = pd.read_csv(


In [21]:
# Next I want to filter the weather data to only look at sunday, and I will divide the data based on the "Time of day"

hour = weather_df["period_string"].dt.hour

def sunday_time_bin(h):
    if 4 <= h < 8:
        return 0   
    elif 8 <= h < 12:
        return 1  
    elif 12 <= h < 19:
        return 2   
    elif 19 <= h < 22:
        return 3   
    else:
        return 4   # overnight (Sunday)


weather_df["time_period"] = hour.map(sunday_time_bin).astype("int64")



In [ ]:
# Next I will merge the weather data into chunks based on which time period it is in 
weather_sun = weather_df[weather_df["period_string"].dt.dayofweek == 6].copy()

weather_sun["date"] = weather_sun["period_string"].dt.floor("D")

# We could take some kind of average/aggregation of the weather data for each window, but since this is weather forecasting data, I will just take the first forecast entry in each time window as the forecast for the entire time 

# Sort by time so that we can choose the first entry

weather_clip = (
    weather_sun
    .sort_values(["date", "time_period", "period_string"])
    .drop_duplicates(subset=["date", "time_period"], keep="first")
)

In [23]:
# Now lets merge the two together
sun_df["time_period"] = sun_df["Time of Day (morning: 0-late evening: 4)*"].astype("int64")
weather_clip["time_period"] = weather_clip["time_period"].astype("int64")

merged = sun_df.merge(
    weather_clip,
    on=["date", "time_period"],
    how="left",
    suffixes=("", "_weather")
)
# drop redundant column
merged = merged.drop(columns=["Time of Day (morning: 0-late evening: 4)*"])

In [24]:
# Load into a new csv file for each time period with the weather merged with the summary and schedule data 
loc = '../data/clean_data/'
for p in range(5):
    to_csv(
        merged[merged["time_period"] == p],
        loc + f"schedule_weather_sunday_period_{p}.csv"
    )